## Create annotated PDB interfaces TSV
Annotations have been gathered from various sources; in this notebook, they processed to create features for the PDB interfaces and combined into one TSV file along with the PDB interface cluster assignments for all 3.12M interfaces.

In [1]:
import numpy as np
import pandas as pd

In [ ]:
# TSV file from collate_cluster_results.py
pdb_clusters = pd.read_csv('/Volumes/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_clusters.tsv', sep='\t')
# Pandas interprets chain name 'NA' as np.nan, replace np.nan values with 'NA'
pdb_clusters['chain_id_0'] = pdb_clusters['chain_id_0'].fillna('NA')
pdb_clusters['chain_id_1'] = pdb_clusters['chain_id_1'].fillna('NA')
pdb_clusters.drop("old", axis=1, inplace=True)
print(pdb_clusters.head())

   old_complex_id chain_id_0          pdb_id chain_id_1  new_complex_id  \
0               0          A  10gs-assembly1          B               0   
1               2          A  117e-assembly1          B               2   
2               3          A  11as-assembly1          B               3   
3               4          A  11ba-assembly1          B               4   
4               5          A  11bg-assembly1          B               5   

   dimercluster  intcluster  dimerrep  intrep  
0        132578        6842         0       0  
1        149449       42136         0       0  
2             0       44306         0       0  
3        189814         369         0       0  
4        189814         369         0       0  


In [ ]:
# TSV file from residue_mapping.py
res_map = pd.read_csv('/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/uniprot_residue_mapping.tsv', sep='\t')

In [ ]:
# Define peptide as chain with fewer than 20 resolved residues
res_map.loc[res_map["FLlen"] < 20, "pepStatus"] = 1
res_map.loc[res_map["FLlen"] >= 20, "pepStatus"] = 0

In [5]:
res_map

,chain_index,if_res,new_complex_id,full_id,pdb_id,chain_id,uniprot_id,uniprot_res,FLlen,pepStatus
0,0,"47,48,49,50,57,60,61,62,63,64,65,66,67,68,69,7...",0,DI0_10gs-assembly1_A,10gs,A,P09211,"48,49,50,51,58,61,62,63,64,65,66,67,68,69,70,7...",210,0.0
1,1,"47,48,49,50,60,61,62,63,64,65,66,67,68,69,70,7...",0,DI0_10gs-assembly1_B,10gs,B,P09211,"48,49,50,51,61,62,63,64,65,66,67,68,69,70,71,7...",210,0.0
2,2,"47,48,49,50,57,60,61,62,63,64,65,66,67,68,69,7...",1,DI1_20gs-assembly1_A,20gs,A,P09211,"48,49,50,51,58,61,62,63,64,65,66,67,68,69,70,7...",210,0.0
3,3,"47,48,49,50,57,59,60,61,62,63,64,65,66,67,68,6...",1,DI1_20gs-assembly1_B,20gs,B,P09211,"48,49,50,51,58,60,61,62,63,64,65,66,67,68,69,7...",210,0.0
4,4,"51,52,84,85,86,87,90,124,125,126,127,128,129,1...",2,DI2_117e-assembly1_A,117e,A,P00817,"52,53,85,86,87,88,91,125,126,127,128,129,130,1...",284,0.0
...,...,...,...,...,...,...,...,...,...,...
6243917,6243917,"41,42,44,45,46,47,48,49,50,51,52,53,54,55,56,5...",3121958,DI3121958_8zz0-assembly1_E,8zz0,E,O06873,"41,42,44,45,46,47,48,49,50,51,52,53,54,55,56,5...",215,0.0
6243918,6243918,"66,176,179,180,183,184,186,187,188,189,190,191...",3121959,DI3121959_8zz0-assembly1_E,8zz0,E,O06873,"66,176,179,180,183,184,186,187,188,189,190,191...",215,0.0
6243919,6243919,"41,42,44,45,46,47,48,49,50,51,52,53,54,55,56,5...",3121959,DI3121959_8zz0-assembly1_F,8zz0,F,O06873,"41,42,44,45,46,47,48,49,50,51,52,53,54,55,56,5...",232,0.0
6243920,6243920,"3,4,5,7,8,9,10,11,12,13,14,15,17,18,21,173,174...",3121960,DI3121960_8zz0-assembly1_F,8zz0,F,O06873,"3,4,5,7,8,9,10,11,12,13,14,15,17,18,21,173,174...",232,0.0


In [6]:
def get_if_len(r):
    ifres = [int(x) for x in str(r["if_res"]).split(",") if x != 'nan']
    iflen = len(ifres)
    return iflen

def get_dimer_len(x):
    dimer_len = sum(x["FLlen"])
    return dimer_len

def get_protpep_status(x):
    protpep_sum = sum(x["pepStatus"])
    if protpep_sum == 0:
        return "Protein-protein"
    elif protpep_sum == 1:
        return "Protein-peptide"
    elif protpep_sum == 2:
        return "Peptide-peptide"
    else:
        return "Unclassified"

def get_uniprots(x):
    uniprots = [str(y) for y in x["uniprot_id"] if ((y != "id not found") & (y != "chimeric"))]
    uniprot_string = "-".join(sorted(uniprots))
    return uniprot_string


In [ ]:
# Number of interface residues extracted by Foldseek-Interface for each chain
res_map["if_len"] = res_map.apply(lambda x: get_if_len(x), axis=1)
# Total length of dimer (not just interface)
dimer_len = res_map.groupby("new_complex_id").apply(get_dimer_len).to_frame()
dimer_len.columns = ["dimer_len"]
# Combine Uniprot IDs for each interface
uniprots = res_map.groupby("new_complex_id").apply(get_uniprots).to_frame()
uniprots.columns = ["uniprot_ids"]
# Combine protein/peptide information for each interface
protpep_status = res_map.groupby("new_complex_id").apply(get_protpep_status).to_frame()
protpep_status.columns = ["protpep_status"]
res_map['idx'] = res_map.groupby('new_complex_id').cumcount()
if_lens = res_map.pivot(index='new_complex_id', columns='idx')[['if_len']]
if_lens = if_lens.sort_index(axis=1, level=1)
if_lens.columns = [f'{x}_{y}' for x,y in if_lens.columns]
if_lens = if_lens.reset_index()
pdb_clusters["if_len_0"] = pdb_clusters["new_complex_id"].map(dict(zip(if_lens.new_complex_id, if_lens.if_len_0)))
pdb_clusters["if_len_1"] = pdb_clusters["new_complex_id"].map(dict(zip(if_lens.new_complex_id, if_lens.if_len_1)))
pdb_clusters["dimer_len"] = pdb_clusters["new_complex_id"].map(dict(zip(dimer_len.index, dimer_len.dimer_len)))
pdb_clusters["uniprot_ids"] = pdb_clusters["new_complex_id"].map(dict(zip(uniprots.index, uniprots.uniprot_ids)))
pdb_clusters["protpep_status"] = pdb_clusters["new_complex_id"].map(dict(zip(protpep_status.index, protpep_status.protpep_status)))

/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_19022/1071549693.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dimer_len = res_map.groupby("new_complex_id").apply(get_dimer_len).to_frame()
/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_19022/1071549693.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  uniprots = res_map.groupby("new_complex_id").apply(get_uniprots).to_frame()


In [8]:
pdb_clusters["maxiflen"] = pdb_clusters[["if_len_0","if_len_1"]].apply(lambda x: np.nanmax(x), axis=1)
pdb_clusters["miniflen"] = pdb_clusters[["if_len_0","if_len_1"]].apply(lambda x: np.nanmin(x), axis=1)

In [ ]:
# Group interfaces into quartiles based on the maximum interface chain length
pdb_clusters_dimeronly = pdb_clusters[pdb_clusters["dimerrep"] == 1]
print(pdb_clusters_dimeronly.maxiflen.describe())
pdb_clusters_intonly = pdb_clusters[pdb_clusters["intrep"] == 1]
print(pdb_clusters_intonly.maxiflen.describe())
pdb_clusters.loc[pdb_clusters["maxiflen"] < 12, "if_size_cat"] = "X-Small"
pdb_clusters.loc[((pdb_clusters["maxiflen"] >= 12) & (pdb_clusters["maxiflen"] < 21)), "if_size_cat"] = "Small"
pdb_clusters.loc[((pdb_clusters["maxiflen"] >= 21) & (pdb_clusters["maxiflen"] < 36)), "if_size_cat"] = "Medium"
pdb_clusters.loc[pdb_clusters["maxiflen"] >= 36, "if_size_cat"] = "Large"              

count    189830.000000
mean         28.724327
std          29.645250
min           4.000000
25%          11.000000
50%          20.000000
75%          37.000000
max        1053.000000
Name: maxiflen, dtype: float64
count    77167.000000
mean        28.752316
std         27.718936
min          4.000000
25%         12.000000
50%         21.000000
75%         36.000000
max       1018.000000
Name: maxiflen, dtype: float64


In [10]:
pdb_clusters

,old_complex_id,chain_id_0,pdb_id,chain_id_1,new_complex_id,dimercluster,intcluster,dimerrep,intrep,if_len_0,if_len_1,dimer_len,uniprot_ids,protpep_status,maxiflen,miniflen,if_size_cat
0,0,A,10gs-assembly1,B,0,132578,6842,0,0,41,40,420,P09211-P09211,Protein-protein,41,40,Large
1,2,A,117e-assembly1,B,2,149449,42136,0,0,28,28,568,P00817-P00817,Protein-protein,28,28,Medium
2,3,A,11as-assembly1,B,3,0,44306,0,0,54,57,658,P00963-P00963,Protein-protein,57,54,Large
3,4,A,11ba-assembly1,B,4,189814,369,0,0,59,59,252,P00669-P00669,Protein-protein,59,59,Large
4,5,A,11bg-assembly1,B,5,189814,369,0,0,59,56,252,P00669-P00669,Protein-protein,59,56,Large
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3121956,249407654,A,9xim-assembly1,C,2794796,182167,76311,0,0,132,132,787,P12851-P12851,Protein-protein,132,132,Large
3121957,249407655,A,9xim-assembly1,D,2794797,182168,76312,0,0,63,64,788,P12851-P12851,Protein-protein,64,63,Large
3121958,249407656,B,9xim-assembly1,C,2794798,182168,76312,0,0,64,65,787,P12851-P12851,Protein-protein,65,64,Large
3121959,249407657,B,9xim-assembly1,D,2794799,182167,76311,0,0,132,131,788,P12851-P12851,Protein-protein,132,131,Large


## Add coiled-coil annotations

In [ ]:
# From get_coiledcoils.py
coils = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/uniprot_coiledcoils.tsv", sep="\t")
coils.dropna(subset="ft_coiled", axis=0, inplace=True)
print(coils.shape[0])
coils["range"] = [x.split("COILED")[-1].split(";")[0] for x in coils["ft_coiled"]]
coils

2590


,uniprot_id,ft_coiled,range
2,P37296,"COILED 113..154; /evidence=""ECO:0000255""; COIL...",297..347
13,Q8RD95,"COILED 125..152; /evidence=""ECO:0000256|SAM:Co...",125..152
20,Q9HAU5,"COILED 54..134; /evidence=""ECO:0000255""; COILE...",487..559
61,Q2YD98,"COILED 165..199; /evidence=""ECO:0000255""",165..199
173,A0A0H2ZMB9,"COILED 327..380; /evidence=""ECO:0000256|SAM:Co...",327..380
...,...,...,...
51674,G0S292,"COILED 117..151; /evidence=""ECO:0000256|SAM:Co...",117..151
51759,A0A5H1ZR46,"COILED 135..162; /evidence=""ECO:0000256|SAM:Co...",135..162
51766,P46863,"COILED 362..462; /evidence=""ECO:0000255""; COIL...",889..918
51795,A0A0J9X267,"COILED 56..83; /evidence=""ECO:0000256|SAM:Coils""",56..83


In [ ]:
# Listed coiled-coil domain range must overlap with at least one interface residue
def check_coil_range(r):
    coil_in_range = np.nan
    if isinstance(r["uniprot_res"], str):
        full_range = [int(x) for x in str(r['coil_range']).split('..')]
        if len(full_range) < 2:
            return 0
        coil_in_range = 0
        for ifres in [int(x) for x in r['uniprot_res'].split(',') if ((x != 'nan') & (x != ''))]:
            if ifres in full_range:
                coil_in_range = 1
                break
    return coil_in_range

def return_coil_sum(x):
    return sum(x["coil_in_range"])

res_map["coil_range"] = res_map["uniprot_id"].map(dict(zip(coils.uniprot_id, coils.range)))
res_map_coilsonly = res_map.dropna(subset="coil_range", axis=0)
#res_map.drop("range", axis=1, inplace=True)
res_map_coilsonly["coil_in_range"] = res_map_coilsonly.apply(lambda x: check_coil_range(x), axis=1)

coil_list = res_map_coilsonly.groupby("new_complex_id").apply(lambda x: return_coil_sum(x), include_groups=False)
coil_list = coil_list.to_frame()
coil_list.columns = ["coil_status"]
coil_list[coil_list["coil_status"] == 2]

/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_19022/3159250857.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_map_coilsonly["coil_in_range"] = res_map_coilsonly.apply(lambda x: check_coil_range(x), axis=1)


,coil_status
new_complex_id,
4289,2.0
4290,2.0
8247,2.0
14347,2.0
20498,2.0
...,...
3119720,2.0
3119724,2.0
3119727,2.0


In [ ]:
# If both chains have a coiled-coil domain at the interface, the interaction is labeled "Coiled-coil", otherwise not
pdb_clusters = pd.merge(pdb_clusters, coil_list, on="new_complex_id", how="left")
pdb_clusters.loc[pdb_clusters["coil_status"] != 2, "coil_status"] = "Not coiled-coil"
pdb_clusters.loc[pdb_clusters["coil_status"] == 2, "coil_status"] = "Coiled-coil"

/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_19022/799314029.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Not coiled-coil' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  pdb_clusters.loc[pdb_clusters["coil_status"] != 2, "coil_status"] = "Not coiled-coil"


In [14]:
pdb_clusters[pdb_clusters["coil_status"] == "Coiled-coil"]

,old_complex_id,chain_id_0,pdb_id,chain_id_1,new_complex_id,dimercluster,intcluster,dimerrep,intrep,if_len_0,if_len_1,dimer_len,uniprot_ids,protpep_status,maxiflen,miniflen,if_size_cat,coil_status
1362,605251,A,1aa0-assembly1,A-2,20498,65585,62662,0,0,63,65,230,P10104-P10104,Protein-protein,65,63,Large,Coiled-coil
1363,605252,A,1aa0-assembly1,A-3,20499,65585,62662,0,0,65,63,230,P10104-P10104,Protein-protein,65,63,Large,Coiled-coil
1364,605253,A-2,1aa0-assembly1,A-3,20500,65585,62662,1,0,63,65,230,P10104-P10104,Protein-protein,65,63,Large,Coiled-coil
5790,2552542,A,1aq5-assembly1,B,82524,130947,30999,0,0,29,30,98,P05099-P05099,Protein-protein,30,29,Medium,Coiled-coil
5791,2552543,A,1aq5-assembly1,C,82525,130947,30999,0,0,30,29,98,P05099-P05099,Protein-protein,30,29,Medium,Coiled-coil
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3120218,91034434,q,9l5t-assembly1,t,1355626,93089,22763,1,0,58,63,284,G0SFY0-G0SFY0,Protein-protein,63,58,Large,Coiled-coil
3120219,91034481,r,9l5t-assembly1,s,1355631,93094,22763,1,0,60,53,287,G0SFY0-G0SFY0,Protein-protein,60,53,Large,Coiled-coil
3120904,115180398,C,9mhf-assembly1,D,1492479,186691,33981,1,1,140,135,574,Q14457-Q6ZNE5,Protein-protein,140,135,Large,Coiled-coil
3120911,115180408,C,9mhg-assembly1,D,1492486,186691,33981,0,0,148,144,610,Q14457-Q6ZNE5,Protein-protein,148,144,Large,Coiled-coil


In [15]:
print(pdb_clusters.shape[0])

3121961


## Add Pfam annotations

In [ ]:
# Listed Pfam domain range must overlap with at least one interface residue
def check_pfam_range(r):
    in_pfam = 0
    full_range = range(int(r["start"]), int(r["end"]) + 1)
    for ifres in [int(x) for x in r['if_res'].split(',')]:
        if ifres in full_range:
            in_pfam = 1
    return in_pfam

# All Pfam domains at the interface for a given chain are joined by "," in alphabetical order
def get_pfam_list(x):
    return ",".join(sorted([y for y in x["pfam_name"] if ((y != "") & (y is not None))]))

In [ ]:
# From get_pfam.py
pfam = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/pdb_pfam.tsv", sep="\t")
res_map_pluspfam = pd.merge(res_map, pfam, on=['pdb_id','chain_id'], how='left')
res_map_pluspfam.dropna(subset=["start","end"], axis=0, inplace=True)
res_map_pluspfam['pfam_in_range'] = res_map_pluspfam.apply(lambda x: check_pfam_range(x), axis=1)
res_map_pluspfam.loc[res_map_pluspfam['pfam_in_range'] == 0, 'pfam_id'] = None
res_map_pluspfam.loc[res_map_pluspfam['pfam_in_range'] == 0, 'pfam_name'] = None
res_map_pluspfam.drop_duplicates(subset=['full_id','pfam_id'], keep="first", inplace=True)

In [18]:
pfam_list = res_map_pluspfam.groupby(["new_complex_id", "full_id"]).apply(lambda x: get_pfam_list(x), include_groups=False)
pfam_list = pd.DataFrame(pfam_list)
pfam_list.columns = ["pfam_list"]
pfam_list["num_pfam_domains"] = [len(x.split(",")) if x != '' else 0 for x in pfam_list.pfam_list]

In [19]:
pfam_list["idx"] = pfam_list.groupby("new_complex_id")["pfam_list"].cumcount()
pfam_list.reset_index(inplace=True)
pfam_list = pfam_list.pivot(index='new_complex_id', columns='idx')[['pfam_list','num_pfam_domains']]
pfam_list = pfam_list.sort_index(axis=1, level=1)
pfam_list.columns = [f'{x}_{y}' for x,y in pfam_list.columns]
pfam_list = pfam_list.reset_index()
pfam_list

,new_complex_id,num_pfam_domains_0,pfam_list_0,num_pfam_domains_1,pfam_list_1
0,0,4.0,"Glutathione S-transferase, C-terminal domain,G...",4.0,"Glutathione S-transferase, C-terminal domain,G..."
1,1,4.0,"Glutathione S-transferase, C-terminal domain,G...",4.0,"Glutathione S-transferase, C-terminal domain,G..."
2,2,1.0,Inorganic pyrophosphatase,1.0,Inorganic pyrophosphatase
3,3,1.0,Aspartate-ammonia ligase,1.0,Aspartate-ammonia ligase
4,4,1.0,Pancreatic ribonuclease,1.0,Pancreatic ribonuclease
...,...,...,...,...,...
2671776,3121956,1.0,MotA/TolQ/ExbB proton channel family,1.0,MotA/TolQ/ExbB proton channel family
2671777,3121957,1.0,MotA/TolQ/ExbB proton channel family,1.0,MotA/TolQ/ExbB proton channel family
2671778,3121958,1.0,MotA/TolQ/ExbB proton channel family,1.0,MotA/TolQ/ExbB proton channel family
2671779,3121959,1.0,MotA/TolQ/ExbB proton channel family,1.0,MotA/TolQ/ExbB proton channel family


In [ ]:
# Pfam domain lists for the two chains of an interface are joined by "_" in alphabetical order
# Mark which interfaces have exactly one Pfam domain on each interface chain, this will help with analysis later
pfam_list.loc[((pfam_list["num_pfam_domains_0"]  >= 1) & (pfam_list["num_pfam_domains_1"] >= 1)),"pfam_pair"] = pfam_list.loc[((pfam_list["num_pfam_domains_0"] >= 1) & (pfam_list["num_pfam_domains_1"] >= 1)),["pfam_list_0","pfam_list_1"]].apply(lambda x: "_".join(sorted([str(y) for y in x if y is not None])), axis=1)
pfam_list.loc[((pfam_list["num_pfam_domains_0"] == 1) & ((pfam_list["num_pfam_domains_1"] == 0) | (pfam_list["num_pfam_domains_1"].isna()))), "pfam_pair"] = pfam_list.loc[((pfam_list["num_pfam_domains_0"] == 1) & ((pfam_list["num_pfam_domains_1"] == 0) | (pfam_list["num_pfam_domains_1"].isna()))), "pfam_list_0"]
pfam_list.loc[(((pfam_list["num_pfam_domains_0"] == 0) | (pfam_list["num_pfam_domains_0"].isna())) & (pfam_list["num_pfam_domains_1"] == 1)), "pfam_pair"] = pfam_list.loc[(((pfam_list["num_pfam_domains_0"] == 0) | (pfam_list["num_pfam_domains_0"].isna())) & (pfam_list["num_pfam_domains_1"] == 1)), "pfam_list_1"]
pfam_list.loc[((pfam_list["num_pfam_domains_0"]  == 1) & (pfam_list["num_pfam_domains_1"] == 1)),"pfam_is_pair"] = 1
pfam_list["pfam_is_pair"] = pfam_list["pfam_is_pair"].fillna(0)
pdb_clusters.loc[:,'pfam_annot'] = pdb_clusters['new_complex_id'].map(dict(zip(pfam_list['new_complex_id'], pfam_list['pfam_pair'])))
pdb_clusters.loc[:,'pfam_is_pair'] = pdb_clusters['new_complex_id'].map(dict(zip(pfam_list['new_complex_id'], pfam_list['pfam_is_pair'])))

In [21]:
res_map_pluspfam[res_map_pluspfam["new_complex_id"] == 1370]

,chain_index,if_res,new_complex_id,full_id,pdb_id,chain_id,uniprot_id,uniprot_res,FLlen,pepStatus,if_len,idx,coil_range,start,end,pfam_name,pfam_id,pfam_in_range
3516,2740,"62,63,64,65,66,67,68,69,70,71,72,90,94,95,96",1370,DI1370_1a17-assembly1_A,1a17,A,P53041,"77,78,79,80,81,82,83,84,85,86,87,105,109,110,111",161,0.0,15,0,NaN,13.0,46.0,None,None,0
3518,2741,"62,63,64,65,66,67,68,69,70,71,72,90,94,95,96",1370,DI1370_1a17-assembly1_A-4,1a17,A,P53041,"77,78,79,80,81,82,83,84,85,86,87,105,109,110,111",161,0.0,15,1,NaN,13.0,46.0,None,None,0


In [22]:
pdb_clusters[pdb_clusters["pfam_annot"].isna()]

,old_complex_id,chain_id_0,pdb_id,chain_id_1,new_complex_id,dimercluster,intcluster,dimerrep,intrep,if_len_0,if_len_1,dimer_len,uniprot_ids,protpep_status,maxiflen,miniflen,if_size_cat,coil_status,pfam_annot,pfam_is_pair
14,22,L,12e8-assembly1,H,22,21854,40398,0,0,75,68,439,nan-nan,Protein-protein,75,68,Large,Not coiled-coil,NaN,NaN
15,23,M,12e8-assembly2,P,23,185924,40398,0,0,73,62,439,nan-nan,Protein-protein,73,62,Large,Not coiled-coil,NaN,NaN
23,43,D,173d-assembly1,D-2,43,2,3001,1,1,5,5,14,nan-nan,Peptide-peptide,5,5,X-Small,Not coiled-coil,NaN,NaN
24,44,C,173d-assembly2,C-3,44,3,3001,1,0,5,5,14,nan-nan,Peptide-peptide,5,5,X-Small,Not coiled-coil,NaN,NaN
139,8793,A,1a17-assembly1,A-4,1370,238,76818,0,0,15,15,322,P53041-P53041,Protein-protein,15,15,Small,Not coiled-coil,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3121890,144465953,A-8,9nek-assembly1,A-9,1623091,159506,55539,0,0,74,76,702,A0A6B9D5B5-A0A6B9D5B5,Protein-protein,76,74,Large,Not coiled-coil,NaN,NaN
3121891,144466005,A-9,9nek-assembly1,A-10,1623096,159506,55539,0,0,74,76,702,A0A6B9D5B5-A0A6B9D5B5,Protein-protein,76,74,Large,Not coiled-coil,NaN,NaN
3121892,144466027,A-9,9nek-assembly1,A-32,1623097,159505,55538,0,0,48,54,702,A0A6B9D5B5-A0A6B9D5B5,Protein-protein,54,48,Large,Not coiled-coil,NaN,NaN
3121893,144466052,A-9,9nek-assembly1,A-57,1623098,159504,55537,1,1,22,22,702,A0A6B9D5B5-A0A6B9D5B5,Protein-protein,22,22,Medium,Not coiled-coil,NaN,NaN


In [23]:
pdb_clusters[pdb_clusters["pfam_is_pair"] == 1].shape[0]

1682703

In [24]:
print(pdb_clusters.shape[0])

3121961


## Add interface structure

In [ ]:
# From get_dssp.py
if_struct_types = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/if_struct_types.tsv", sep="\t")
if_struct_types

,new_complex_id,chain_id,helix_frac,beta_frac,turn_frac,bend_frac,other_frac
0,2180585,A,0.465116,0.058140,0.127907,0.104651,0.244186
1,2180585,B,0.265823,0.063291,0.189873,0.139241,0.341772
2,2180586,A,0.090909,0.363636,0.181818,0.090909,0.272727
3,2180586,C,1.000000,0.000000,0.000000,0.000000,0.000000
4,2180587,B,0.378788,0.212121,0.166667,0.015152,0.227273
...,...,...,...,...,...,...,...
6052049,1933784,C,0.200000,0.000000,0.200000,0.200000,0.400000
6052050,1933785,A,0.300000,0.400000,0.200000,0.000000,0.100000
6052051,1933785,D,0.300000,0.000000,0.200000,0.100000,0.400000
6052052,1933786,A,0.538462,0.076923,0.038462,0.076923,0.269231


In [ ]:
# Create weighted averages of secondary structure elements for the entire interface
if_struct_types['idx'] = if_struct_types.groupby('new_complex_id').cumcount()
if_struct_types = if_struct_types.pivot(index='new_complex_id', columns='idx')[['helix_frac','beta_frac','bend_frac','turn_frac','other_frac']]
if_struct_types = if_struct_types.sort_index(axis=1, level=1)
if_struct_types.columns = [f'{x}_{y}' for x,y in if_struct_types.columns]
if_struct_types = if_struct_types.reset_index()
if_struct_types["if_len_0"] = if_struct_types["new_complex_id"].map(dict(zip(pdb_clusters.new_complex_id, pdb_clusters.if_len_0)))
if_struct_types["if_len_1"] = if_struct_types["new_complex_id"].map(dict(zip(pdb_clusters.new_complex_id, pdb_clusters.if_len_1)))
if_struct_types["avg_helix_frac"] = (if_struct_types["if_len_0"] * if_struct_types["helix_frac_0"] + if_struct_types["if_len_1"] * if_struct_types["helix_frac_1"]) / (if_struct_types["if_len_0"] + if_struct_types["if_len_1"])
if_struct_types["avg_beta_frac"] = (if_struct_types["if_len_0"] * if_struct_types["beta_frac_0"] + if_struct_types["if_len_1"] * if_struct_types["beta_frac_1"]) / (if_struct_types["if_len_0"] + if_struct_types["if_len_1"])
if_struct_types["avg_bend_frac"] = (if_struct_types["if_len_0"] * if_struct_types["bend_frac_0"] + if_struct_types["if_len_1"] * if_struct_types["bend_frac_1"]) / (if_struct_types["if_len_0"] + if_struct_types["if_len_1"])
if_struct_types["avg_turn_frac"] = (if_struct_types["if_len_0"] * if_struct_types["turn_frac_0"] + if_struct_types["if_len_1"] * if_struct_types["turn_frac_1"]) / (if_struct_types["if_len_0"] + if_struct_types["if_len_1"])
if_struct_types["avg_other_frac"] = (if_struct_types["if_len_0"] * if_struct_types["other_frac_0"] + if_struct_types["if_len_1"] * if_struct_types["other_frac_1"]) / (if_struct_types["if_len_0"] + if_struct_types["if_len_1"])

In [27]:
pdb_clusters["helix_frac"] = pdb_clusters["new_complex_id"].map(dict(zip(if_struct_types.new_complex_id, if_struct_types.avg_helix_frac)))
pdb_clusters["beta_frac"] = pdb_clusters["new_complex_id"].map(dict(zip(if_struct_types.new_complex_id, if_struct_types.avg_beta_frac)))
pdb_clusters["bend_frac"] = pdb_clusters["new_complex_id"].map(dict(zip(if_struct_types.new_complex_id, if_struct_types.avg_bend_frac)))
pdb_clusters["turn_frac"] = pdb_clusters["new_complex_id"].map(dict(zip(if_struct_types.new_complex_id, if_struct_types.avg_turn_frac)))
pdb_clusters["other_frac"] = pdb_clusters["new_complex_id"].map(dict(zip(if_struct_types.new_complex_id, if_struct_types.avg_other_frac)))

In [ ]:
# 'Bend' and 'turn' are combined into one category
pdb_clusters["bendturn_frac"] = pdb_clusters["bend_frac"] + pdb_clusters["turn_frac"]
# Not every residue has a DSSP assignment, so the interface residues without any labeled secondary structure element are considered 'unassigned'
pdb_clusters["unassign_frac"] = 1 - pdb_clusters["helix_frac"] - pdb_clusters["beta_frac"] - pdb_clusters["bendturn_frac"]
pdb_clusters_nossna = pdb_clusters.dropna(subset=["helix_frac","beta_frac","bend_frac","turn_frac"], axis=0, how='all')
print(pdb_clusters_nossna.shape[0])

pdb_clusters_nossna["ss_major"] = pdb_clusters_nossna[["helix_frac", "beta_frac", "bendturn_frac", "unassign_frac"]].apply('idxmax', axis=1)

pdb_clusters["ss_major"] = pdb_clusters["old_complex_id"].map(dict(zip(pdb_clusters_nossna.old_complex_id, pdb_clusters_nossna.ss_major)))
pdb_clusters["ss_major"].value_counts()

2984507


/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_19022/260723380.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pdb_clusters_nossna["ss_major"] = pdb_clusters_nossna[["helix_frac", "beta_frac", "bendturn_frac", "unassign_frac"]].apply('idxmax', axis=1)


ss_major
helix_frac       1106384
bendturn_frac     685418
unassign_frac     600099
beta_frac         592606
Name: count, dtype: int64

In [ ]:
# Majority element of the interface
pdb_clusters.loc[pdb_clusters["ss_major"] == "helix_frac", "ss_major"] = "Helix"
pdb_clusters.loc[pdb_clusters["ss_major"] == "beta_frac", "ss_major"] = "Strand"
pdb_clusters.loc[pdb_clusters["ss_major"] == "bendturn_frac", "ss_major"] = "Turn"
pdb_clusters.loc[pdb_clusters["ss_major"] == "unassign_frac", "ss_major"] = "Unassigned"
pdb_clusters.loc[pdb_clusters["ss_major"].isna(), "ss_major"] = "NA"

In [30]:
print(pdb_clusters.shape[0])

3121961


## Add taxonomy

In [ ]:
# From PDB metadata
pdb_mapping_lookup = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/pdb.lookup", header=None, sep="\t")
pdb_mapping_lookup.columns = ['chain_index','full_id','complex_id']
pdb_mapping_lookup['pdb_id'] = [x.split("_")[0] for x in pdb_mapping_lookup.full_id]
pdb_mapping_lookup['chain_id'] = [x.split("_")[-1] for x in pdb_mapping_lookup.full_id]
pdb_mapping_taxid = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/pdb_seq_mapping", header=None, sep=" ")
pdb_mapping_taxid.columns = ['chain_index','tax_id']
pdb_mapping_lookup['tax_id'] = pdb_mapping_lookup['chain_index'].map(dict(zip(pdb_mapping_taxid.chain_index, pdb_mapping_taxid.tax_id)))
pdb_mapping_lookup.drop_duplicates(subset=['full_id','tax_id'], keep='first', inplace=True)
pdb_mapping_lookup.loc[pdb_mapping_lookup['tax_id'] == 0, 'tax_id'] = np.nan
pdb_mapping_lookup.head()

,chain_index,full_id,complex_id,pdb_id,chain_id,tax_id
0,0,200l-assembly1_A,0,200l-assembly1,A,10665.0
1,1,101m-assembly1_A,1,101m-assembly1,A,9755.0
2,2,201l-assembly1_A,2,201l-assembly1,A,10665.0
3,3,201l-assembly2_B,3,201l-assembly2,B,10665.0
4,4,102l-assembly1_A,4,102l-assembly1,A,10665.0


In [32]:
pdb_clusters['full_id'] = pdb_clusters[['pdb_id','chain_id_0']].apply(lambda x: "_".join(x), axis=1)
pdb_clusters['tax_id_0'] = pdb_clusters['full_id'].map(dict(zip(pdb_mapping_lookup.full_id, pdb_mapping_lookup.tax_id)))
pdb_clusters['full_id'] = pdb_clusters[['pdb_id','chain_id_1']].apply(lambda x: "_".join(x), axis=1)
pdb_clusters['tax_id_1'] = pdb_clusters['full_id'].map(dict(zip(pdb_mapping_lookup.full_id, pdb_mapping_lookup.tax_id)))
num_tax_annot = pdb_clusters[((~pdb_clusters['tax_id_0'].isna()) & (~pdb_clusters['tax_id_1'].isna()))].shape[0]
num_no_annot = pdb_clusters[((pdb_clusters['tax_id_0'].isna()) & (pdb_clusters['tax_id_1'].isna()))].shape[0]
print("Number of interfaces with two species annotated: ", num_tax_annot, f"({round(num_tax_annot/3121961*100, 1)}%)")
print("Number of interfaces with no species annotated: ", num_no_annot, f"({round(num_no_annot/3121961*100, 1)}%)")

Number of interfaces with two species annotated:  3019201 (96.7%)
Number of interfaces with no species annotated:  95705 (3.1%)


In [33]:
intracluster_tax_summary = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/taxid_intraspecies_all.tsv", sep="\t")
intracluster_tax_summary

,taxid,category
0,11,Bacteria
1,17,Bacteria
2,33,Bacteria
3,34,Bacteria
4,38,Bacteria
...,...,...
5382,3052230,Viruses
5383,3052342,Viruses
5384,3052465,Viruses
5385,3067671,Viruses


In [34]:
pdb_clusters["tax_category"] = pdb_clusters["tax_id_0"].map(dict(zip(intracluster_tax_summary.taxid, intracluster_tax_summary.category)))
pdb_clusters.loc[(pdb_clusters["tax_id_0"] != pdb_clusters["tax_id_1"]), "tax_category"] = "Interspecies"

In [35]:
print(pdb_clusters.shape[0])

3121961


## Add disorder

In [ ]:
# From get_disorder.py
disfracs = pd.read_csv('/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/disorder_fractions.tsv', sep='\t')
print(disfracs.shape[0])
# Issue retrieving IUPred2A results for P03436 programatically, but the entry for this Uniprot on the IUPred server shows no predicted disorder anywhere in the protein
# Manually replace the NaN values for this Uniprot with entirely ordered fraction
disfracs.loc[disfracs["uniprot_id"] == "P03436", "disfrac"] = 0.0
disfracs.drop_duplicates(subset=['new_complex_id','chain_id'], inplace=True, keep=False)
print(disfracs.shape[0])
disfracs.dropna(subset=['disfrac'], inplace=True, axis=0)
print(disfracs.shape[0])
disfracs['idx'] = disfracs.groupby('new_complex_id').cumcount()
print(disfracs[disfracs['idx'] > 1])
disfracs = disfracs.pivot(index='new_complex_id', columns='idx')['disfrac']
disfracs = disfracs.sort_index(axis=1, level=1)
disfracs.columns = ['disfrac_0','disfrac_1']
disfracs = disfracs.reset_index()
disfracs.dropna(subset=['disfrac_0','disfrac_1'], inplace=True, axis=0)
print(disfracs.shape[0])
disfracs['maxdisfrac'] = np.nanmax(disfracs[['disfrac_0','disfrac_1']].values, axis=1)
disfracs['mindisfrac'] = np.nanmin(disfracs[['disfrac_0','disfrac_1']].values, axis=1)

5752461
5752461
5746760
Empty DataFrame
Columns: [chain_id, new_complex_id, uniprot_id, disfrac, idx]
Index: []
2800858


In [ ]:
# Find maximum and minimum disorder fractions for the interface
pdb_clusters["maxdisfrac"] = pdb_clusters["new_complex_id"].map(dict(zip(disfracs.new_complex_id, disfracs.maxdisfrac)))
pdb_clusters["mindisfrac"] = pdb_clusters["new_complex_id"].map(dict(zip(disfracs.new_complex_id, disfracs.mindisfrac)))
print(pdb_clusters.head())

   old_complex_id chain_id_0          pdb_id chain_id_1  new_complex_id  \
0               0          A  10gs-assembly1          B               0   
1               2          A  117e-assembly1          B               2   
2               3          A  11as-assembly1          B               3   
3               4          A  11ba-assembly1          B               4   
4               5          A  11bg-assembly1          B               5   

   dimercluster  intcluster  dimerrep  intrep  if_len_0  ...  other_frac  \
0        132578        6842         0       0        41  ...    0.000000   
1        149449       42136         0       0        28  ...    0.295330   
2             0       44306         0       0        54  ...    0.330252   
3        189814         369         0       0        59  ...    0.128141   
4        189814         369         0       0        59  ...    0.131542   

   bendturn_frac unassign_frac    ss_major           full_id  tax_id_0  \
0       0.123457  

In [ ]:
# Categorize interfaces based on the maximum and minimum disorder fractions of the two chains
pdb_clusters.loc[((pdb_clusters['maxdisfrac'] > 0.5) & (pdb_clusters['mindisfrac'] > 0.5)), "if_type"] = "Disorder-disorder"
pdb_clusters.loc[((pdb_clusters['maxdisfrac'] > 0.5) & (pdb_clusters['mindisfrac'] <= 0.5)), "if_type"] = "Disorder-order"
pdb_clusters.loc[((pdb_clusters['maxdisfrac'] <= 0.5) & (pdb_clusters['mindisfrac'] <= 0.5)), "if_type"] = "Order-order"
pdb_clusters['if_type'] = pdb_clusters['if_type'].fillna("Unannotated")
num_order_if = pdb_clusters[pdb_clusters['if_type'] == "Order-order"].shape[0]
num_disord_if = pdb_clusters[pdb_clusters['if_type'] == "Disorder-order"].shape[0]
num_dis_if = pdb_clusters[pdb_clusters['if_type'] == "Disorder-disorder"].shape[0]
num_unchar_if = pdb_clusters[pdb_clusters['if_type'] == "Unannotated"].shape[0]
print("Number of order-order interactions: ", num_order_if, f"({round(num_order_if/3121961*100, 1)}%)")
print("Number of disorder-order interactions: ", num_disord_if, f"({round(num_disord_if/3121961*100, 1)}%)")
print("Number of disorder-disorder interactions: ", num_dis_if, f"({round(num_dis_if/3121961*100, 1)}%)")
print("Number of unannotated interactions: ", num_unchar_if, f"({round(num_unchar_if/3121961*100, 1)}%)")

Number of order-order interactions:  2207574 (70.7%)
Number of disorder-order interactions:  450006 (14.4%)
Number of disorder-disorder interactions:  143278 (4.6%)
Number of unannotated interactions:  321103 (10.3%)


In [39]:
print(pdb_clusters.shape[0])

3121961


## Add CATH

In [ ]:
# Listed CATH domain range must overlap with at least one interface residue
def check_cath_range(r):
    ranges = [x for x in str(r['cath_range']).split(',')]
    full_range = []
    for item in ranges:
        if item == 'nan':
            continue
        curr_range = [int(''.join(filter(str.isdigit, x))) for x in item.split('-') if x != '']
        curr_range = range(curr_range[0],curr_range[1])
        full_range.extend(curr_range)
    in_cath = 0
    for ifres in [int(x) for x in r['if_res'].split(',')]:
        if ifres in full_range:
            in_cath = 1
            break
    return in_cath

def get_cath_list(x):
    return ",".join(sorted([y for y in x["cath_domain"] if y != ""]))

In [ ]:
# Downloaded from CATH server (v4.4)
cath = pd.read_csv('/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/cath-domain-boundaries-seqreschopping.txt', header=None, sep="\t")
cath.columns = ['cath_id', 'cath_range']
cath['pdb_id'] = [x[0:4] for x in cath['cath_id']]
cath['chain_id'] = [x[4] for x in cath['cath_id']]
print(cath.head())

res_map = pd.merge(res_map, cath, on=['pdb_id','chain_id'], how='left')
res_map['cath_in_range'] = res_map.apply(lambda x: check_cath_range(x), axis=1)
res_map.loc[res_map['cath_in_range'] == 0, 'cath_id'] = None
res_map.drop_duplicates(subset=['full_id','cath_id'], keep="first", inplace=True)

   cath_id cath_range pdb_id chain_id
0  101mA00      1-154   101m        A
1  102lA00      1-163   102l        A
2  102mA00      1-154   102m        A
3  103lA00      1-165   103l        A
4  103mA00      1-154   103m        A


In [42]:
cath_ids = []
cath_domains = []
with open("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/cath-domain-list.txt", 'r') as f:
    for line in f.readlines():
        if line[0] == "#":
            continue
        cath_ids.append(line[0:7])
        cath_domain = ".".join([line[12],line[16:20].strip(" "),line[21:26].strip(" "), line[27:32].strip(" ")])
        cath_domains.append(cath_domain)
res_map['cath_domain'] = res_map['cath_id'].map(dict(zip(cath_ids, cath_domains)))
res_map['cath_domain'] = res_map['cath_domain'].fillna("")

In [43]:
cath_list = res_map.groupby(["new_complex_id", "full_id"]).apply(lambda x: get_cath_list(x), include_groups=False)
cath_list = pd.DataFrame(cath_list)
cath_list.columns = ["cath_list"]
#cath_list

In [44]:
cath_list["num_cath_domains"] = [len(x.split(",")) if x != '' else 0 for x in cath_list.cath_list]
#cath_list

In [45]:
cath_list["idx"] = cath_list.groupby("new_complex_id")["cath_list"].cumcount()
cath_list.reset_index(inplace=True)
cath_list = cath_list.pivot(index='new_complex_id', columns='idx')[['cath_list','num_cath_domains']]
cath_list = cath_list.sort_index(axis=1, level=1)
cath_list.columns = [f'{x}_{y}' for x,y in cath_list.columns]
cath_list = cath_list.reset_index()
cath_list

,new_complex_id,cath_list_0,num_cath_domains_0,cath_list_1,num_cath_domains_1
0,0,"1.20.1050.10,3.40.30.10",2,"1.20.1050.10,3.40.30.10",2
1,1,"1.20.1050.10,3.40.30.10",2,"1.20.1050.10,3.40.30.10",2
2,2,3.90.80.10,1,3.90.80.10,1
3,3,3.30.930.10,1,3.30.930.10,1
4,4,3.10.130.10,1,3.10.130.10,1
...,...,...,...,...,...
3121956,3121956,,0,,0
3121957,3121957,,0,,0
3121958,3121958,,0,,0
3121959,3121959,,0,,0


In [46]:
print("Number of interfaces with no CATH domains: ", cath_list[((cath_list["num_cath_domains_0"] == 0) & (cath_list["num_cath_domains_1"] == 0))].shape[0])
print("Number of interfaces with only 1 CATH domain: ", cath_list[(((cath_list["num_cath_domains_0"] == 1) & (cath_list["num_cath_domains_1"] == 0)) | ((cath_list["num_cath_domains_1"] == 1) & (cath_list["num_cath_domains_0"] == 0)))].shape[0])
print("Number of interfaces with exactly 2 CATH domains: ", cath_list[((cath_list["num_cath_domains_0"] == 1) & (cath_list["num_cath_domains_1"] == 1))].shape[0])
print("Number of interfaces with more than 2 CATH annotations: ", cath_list[((cath_list["num_cath_domains_0"] > 1) | (cath_list["num_cath_domains_1"] > 1))].shape[0])

Number of interfaces with no CATH domains:  2157559
Number of interfaces with only 1 CATH domain:  172046
Number of interfaces with exactly 2 CATH domains:  648988
Number of interfaces with more than 2 CATH annotations:  143368


In [47]:
cath_list[((cath_list["num_cath_domains_0"] == 0) & (cath_list["num_cath_domains_1"] == 0))]

,new_complex_id,cath_list_0,num_cath_domains_0,cath_list_1,num_cath_domains_1
43,43,,0,,0
44,44,,0,,0
46,46,,0,,0
109,109,,0,,0
134,134,,0,,0
...,...,...,...,...,...
3121956,3121956,,0,,0
3121957,3121957,,0,,0
3121958,3121958,,0,,0
3121959,3121959,,0,,0


In [ ]:
# CATH domains on a chain are joined with "," in alphabetical order, CATH domain lists for the two chains in an interface are joined with "-" in alphabetical order
# Interfaces with exactly one CATH domain on each chain are marked for further analysis
cath_list.loc[((cath_list["num_cath_domains_0"]  == 1) & (cath_list["num_cath_domains_1"] == 1)),"cath_pair"] = cath_list.loc[((cath_list["num_cath_domains_0"] == 1) & (cath_list["num_cath_domains_1"] == 1)),["cath_list_0","cath_list_1"]].apply(lambda x: "-".join(sorted([str(y) for y in x if y is not None])), axis=1)
cath_list.loc[((cath_list["num_cath_domains_0"] == 1) & (cath_list["num_cath_domains_1"] == 0)), "cath_pair"] = cath_list.loc[((cath_list["num_cath_domains_0"] == 1) & (cath_list["num_cath_domains_1"] == 0)), "cath_list_0"]
cath_list.loc[((cath_list["num_cath_domains_0"] == 0) & (cath_list["num_cath_domains_1"] == 1)), "cath_pair"] = cath_list.loc[((cath_list["num_cath_domains_0"] == 0) & (cath_list["num_cath_domains_1"] == 1)), "cath_list_1"]
cath_list.loc[((cath_list["num_cath_domains_0"]  == 1) & (cath_list["num_cath_domains_1"] == 1)),"cath_is_pair"] = 1
cath_list["cath_is_pair"] = cath_list["cath_is_pair"].fillna(0)
pdb_clusters.loc[:,'cath_annot'] = pdb_clusters['new_complex_id'].map(dict(zip(cath_list['new_complex_id'], cath_list['cath_pair'])))
pdb_clusters.loc[:,'cath_is_pair'] = pdb_clusters['new_complex_id'].map(dict(zip(cath_list['new_complex_id'], cath_list['cath_is_pair'])))

In [49]:
print(pdb_clusters.shape[0])

3121961


## Add antibody information

In [ ]:
# Downloaded from sAbDab server
sabdab = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/sabdab_summary_all.tsv", sep="\t")
sabdab.loc[sabdab["antigen_type"] != "protein", "antigen_chain"] = np.nan
sabdab = sabdab[["pdb","Hchain","Lchain","antigen_chain"]]
sabdab = pd.melt(sabdab, id_vars="pdb", value_vars=["Hchain","Lchain","antigen_chain"])
sabdab.dropna(subset=["value"], axis=0, inplace=True)
sabdab.loc[sabdab["variable"] == "antigen_chain", "variable"] = "Antigen"
sabdab.drop_duplicates(keep="first", inplace=True)
sabdab

,pdb,variable,value
0,9bop,Hchain,A
1,9bop,Hchain,M
2,9bop,Hchain,O
3,9bop,Hchain,C
4,9bop,Hchain,E
...,...,...,...
59658,7tas,Antigen,E
59659,6ejm,Antigen,B
59660,7lo6,Antigen,C
59661,3vi3,Antigen,D


In [51]:
print(sabdab[sabdab.duplicated(subset=["pdb","value"], keep=False)])

Empty DataFrame
Columns: [pdb, variable, value]
Index: []


In [ ]:
# Create interface annotations (combine annotations from the two chains)
pdb_clusters["short_pdb"] = [x[0:4] for x in pdb_clusters["pdb_id"]]
pdb_clusters["short_chain_0"] = [x.split("-")[0] for x in pdb_clusters["chain_id_0"]]
pdb_clusters["short_chain_1"] = [x.split("-")[0] for x in pdb_clusters["chain_id_1"]]
pdb_clusters = pd.merge(pdb_clusters, sabdab, left_on=["short_pdb","short_chain_0"], right_on=["pdb","value"], how="left")
pdb_clusters = pd.merge(pdb_clusters, sabdab, left_on=["short_pdb","short_chain_1"], right_on=["pdb","value"], how="left")
pdb_clusters.loc[pdb_clusters["variable_x"].isna(), "variable_x"] = "Other"
pdb_clusters.loc[pdb_clusters["variable_y"].isna(), "variable_y"] = "Other"
pdb_clusters["antibody"] = pdb_clusters[["variable_x","variable_y"]].apply(lambda x: "-".join(sorted(x)), axis=1)
pdb_clusters["antibody"]

0          Other-Other
1          Other-Other
2          Other-Other
3          Other-Other
4          Other-Other
              ...     
3121956    Other-Other
3121957    Other-Other
3121958    Other-Other
3121959    Other-Other
3121960    Other-Other
Name: antibody, Length: 3121961, dtype: object

In [ ]:
# If neither chain is either antibody or antigen, remove the annotation
pdb_clusters.loc[pdb_clusters["antibody"] == "Other-Other", "antibody"] = "None"
pdb_clusters["antibody"].value_counts()

antibody
None               2937571
Antigen-Other        40394
Hchain-Lchain        34188
Antigen-Hchain       29814
Antigen-Antigen      21777
Antigen-Lchain       20379
Hchain-Other         19527
Lchain-Other         12611
Lchain-Lchain         3160
Hchain-Hchain         2540
Name: count, dtype: int64

In [54]:
print(pdb_clusters.shape[0])

3121961


## Add PDB summary information

In [ ]:
# From get_pdbcharacteristics.py
pdb_keywords = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/pdb_characteristics.tsv", sep="\t")
pdb_keywords

pdb_clusters["pdb_keyword"] = pdb_clusters["short_pdb"].map(dict(zip(pdb_keywords.pdb_id, pdb_keywords.keyword)))
pdb_clusters["pdb_expt_method"] = pdb_clusters["short_pdb"].map(dict(zip(pdb_keywords.pdb_id, pdb_keywords.expt_method)))
pdb_clusters["pdb_resolution"] = pdb_clusters["short_pdb"].map(dict(zip(pdb_keywords.pdb_id, pdb_keywords.resolution)))
pdb_clusters["pdb_title"] = pdb_clusters["short_pdb"].map(dict(zip(pdb_keywords.pdb_id, pdb_keywords.title)))

print("Number of unique PDB keywords:", pdb_clusters["pdb_keyword"].nunique())
print("Description of experimental methods:", pdb_clusters["pdb_expt_method"].value_counts())
print("Average resolution:", pdb_clusters["pdb_resolution"].mean())

Number of unique PDB keywords: 4247
Description of experimental methods: pdb_expt_method
ELECTRON MICROSCOPY         2167756
X-RAY DIFFRACTION            920323
SOLID-STATE NMR               16644
FIBER DIFFRACTION              3544
SOLUTION NMR                   2652
ELECTRON CRYSTALLOGRAPHY       1271
NEUTRON DIFFRACTION             195
POWDER DIFFRACTION               92
SOLUTION SCATTERING              86
Name: count, dtype: int64
Average resolution: 4.266287523002149


In [56]:
print(pdb_clusters.shape[0])

3121961


## Add gene names

In [ ]:
# From get_gene_name_mapping.py
gene_names = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/pdb_gene-name_mapping.tsv", sep="\t")
gene_names.columns = ["pdb_id_genenamedf","chain_id_genenamedf","gene_names"]

In [60]:
pdb_clusters = pd.merge(pdb_clusters, gene_names, left_on=["short_pdb","short_chain_0"], right_on=["pdb_id_genenamedf","chain_id_genenamedf"], how="left")
pdb_clusters.rename(columns={"gene_names":"gene_names_0"}, inplace=True)
pdb_clusters.drop(["pdb_id_genenamedf","chain_id_genenamedf"], axis=1, inplace=True)
pdb_clusters = pd.merge(pdb_clusters, gene_names, left_on=["short_pdb","short_chain_1"], right_on=["pdb_id_genenamedf","chain_id_genenamedf"], how="left")
pdb_clusters.rename(columns={"gene_names":"gene_names_1"}, inplace=True)
pdb_clusters.drop(["pdb_id_genenamedf","chain_id_genenamedf"], axis=1, inplace=True)

In [61]:
print(pdb_clusters.shape[0])

3121961


In [62]:
# Create PDB ID, chain ID, gene name mapping file for server
pdb_clusters_genes_1 = pdb_clusters[["pdb_id","chain_id_0","gene_names_0"]]
pdb_clusters_genes_1.rename(columns={"chain_id_0":"chain_id", "gene_names_0":"gene_names"}, inplace=True)
pdb_clusters_genes_2 = pdb_clusters[["pdb_id","chain_id_1","gene_names_1"]]
pdb_clusters_genes_2.rename(columns={"chain_id_1":"chain_id", "gene_names_1":"gene_names"}, inplace=True)
pdb_clusters_genes = pd.concat([pdb_clusters_genes_1, pdb_clusters_genes_2], axis=0)
pdb_clusters_genes.drop_duplicates(keep="first", inplace=True)
pdb_clusters_genes.to_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/interfaces_gene-names_mapped.tsv", sep="\t", header=None, index=None)

/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_19022/953785324.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pdb_clusters_genes_1.rename(columns={"chain_id_0":"chain_id", "gene_names_0":"gene_names"}, inplace=True)
/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_19022/953785324.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pdb_clusters_genes_2.rename(columns={"chain_id_1":"chain_id", "gene_names_1":"gene_names"}, inplace=True)


## Add GO information

In [ ]:
# Downloaded from UniProtKB Swiss-Prot and TrEMBL
uniprot_go = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/uniprot_go_filtered.tsv", sep="\t", header=None)
uniprot_go.columns = ["uniprot_id","go_cat","evidence"]
uniprot_go

,uniprot_id,go_cat,evidence
0,Q51723,GO:0008422,IEA:TreeGrafter
1,Q51723,GO:0005975,IEA:InterPro
2,Q51760,GO:0046872,IEA:UniProtKB-KW
3,A0A2D0TC98,GO:0051537,IEA:UniProtKB-KW
4,A0A2D0TC98,GO:0051912,IEA:UniProtKB-EC
...,...,...,...
490304,P18541,GO:0044423,IEA:UniProtKB-UniRule
490305,P18541,GO:0003723,IEA:UniProtKB-UniRule
490306,P18541,GO:0008270,IEA:UniProtKB-UniRule
490307,P18541,GO:0046761,IDA:UniProtKB


In [64]:
def get_go_cats(x):
    return ",".join(x["GO"])

res_map_plusgo = res_map
res_map_plusgo["GO"] = res_map_plusgo["uniprot_id"].map(dict(zip(uniprot_go.uniprot_id, uniprot_go.go_cat)))
res_map_plusgo.dropna(subset=["GO"], axis=0, inplace=True)
res_map_plusgo.drop_duplicates(subset=["new_complex_id","GO"], keep="first", inplace=True)
go_cats = res_map_plusgo.groupby("new_complex_id").apply(get_go_cats).to_frame()
go_cats.columns = ["go_cat"]
go_cats

/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_19022/3509939869.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  go_cats = res_map_plusgo.groupby("new_complex_id").apply(get_go_cats).to_frame()


,go_cat
new_complex_id,
0,GO:0006805
1,GO:0006805
2,GO:0006796
3,GO:0070981
4,GO:0050830
...,...
3121956,GO:1902600
3121957,GO:1902600
3121958,GO:1902600


In [65]:
pdb_clusters["go_cat"] = pdb_clusters["new_complex_id"].map(dict(zip(go_cats.index, go_cats.go_cat)))
print(pdb_clusters.shape[0])

3121961


## Clean up

In [66]:
pdb_clusters.columns

Index(['old_complex_id', 'chain_id_0', 'pdb_id', 'chain_id_1',
       'new_complex_id', 'dimercluster', 'intcluster', 'dimerrep', 'intrep',
       'if_len_0', 'if_len_1', 'dimer_len', 'uniprot_ids', 'protpep_status',
       'maxiflen', 'miniflen', 'if_size_cat', 'coil_status', 'pfam_annot',
       'pfam_is_pair', 'helix_frac', 'beta_frac', 'bend_frac', 'turn_frac',
       'other_frac', 'bendturn_frac', 'unassign_frac', 'ss_major', 'full_id',
       'tax_id_0', 'tax_id_1', 'tax_category', 'maxdisfrac', 'mindisfrac',
       'if_type', 'cath_annot', 'cath_is_pair', 'short_pdb', 'short_chain_0',
       'short_chain_1', 'pdb_x', 'variable_x', 'value_x', 'pdb_y',
       'variable_y', 'value_y', 'antibody', 'pdb_keyword', 'pdb_expt_method',
       'pdb_resolution', 'pdb_title', 'gene_names_0', 'gene_names_1',
       'go_cat'],
      dtype='object')

In [67]:
pdb_clusters.drop(["if_len_0","if_len_1","full_id","short_pdb","short_chain_0","short_chain_1","pdb_x","variable_x","value_x","pdb_y","variable_y","value_y"], axis=1, inplace=True)

## Save dataframe with annotations

In [68]:
pdb_clusters.to_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_clusters_annotated.tsv", index=None, sep="\t", float_format="%.3f")